# Module 06 — The Euclidean Algorithm
### Mathematics of Cryptography · Python Companion Notebook

This notebook builds the Euclidean algorithm from the ground up — step trace, table display, step counting, and practical crypto applications.
Every example mirrors Tutorial 06. Run cells top-to-bottom.

| Section | Topic |
|---|---|
| 1 | Setup and display helpers |
| 2 | Division with remainder — the engine |
| 3 | The Euclidean algorithm: clean, trace, and table |
| 4 | Tutorial examples verified |
| 5 | Large numbers — speed demonstration |
| 6 | Step counts and the Fibonacci worst case |
| 7 | Valid affine cipher keys modulo 26 |
| 8 | RSA preview — coprimality at scale |
| 9 | Exercises |
| 10 | Summary and what comes next |

---
## Section 1 — Setup and Display Helpers

In [ ]:
import math
import time

def separator(title=''):
    width = 60
    if title:
        pad = (width - len(title) - 2) // 2
        print('─' * pad + f' {title} ' + '─' * (width - pad - len(title) - 2))
    else:
        print('─' * width)

print('Helpers loaded.')

---
## Section 2 — Division with Remainder: The Engine

The Euclidean algorithm relies on one operation repeated until the remainder hits zero:

$$a = q \cdot b + r \qquad 0 \le r < b$$

In Python: `q, r = divmod(a, b)`. The key insight (proved in Tutorial 06) is:

$$\gcd(a, b) = \gcd(b, r)$$

So we can shrink the problem at every step without changing the answer.

In [ ]:
def show_division(a, b):
    """Show one division step: a = q*b + r."""
    q, r = divmod(a, b)
    print(f'  {a} = {q} · {b} + {r}')
    print(f'  → gcd({a}, {b}) = gcd({b}, {r})')

separator('Single division steps')
for a, b in [(48, 18), (252, 105), (151, 25), (391, 299)]:
    show_division(a, b)
    print()

### Exercise 2.1

Before running, compute the first division step by hand:
1. `a=99, b=78`: what are `q` and `r`?
2. `a=64, b=35`: what are `q` and `r`?

Then verify with `show_division`.

In [ ]:
# Your work here — call show_division(a, b) for each pair


---
## Section 3 — The Euclidean Algorithm: Clean, Trace, and Table

We implement three versions:
- **`gcd(a, b)`** — clean recursive definition, readable as pseudocode
- **`gcd_trace(a, b)`** — prints every step so you can follow the reduction
- **`gcd_table(a, b)`** — tabular format matching Tutorial 06's worked examples

In [ ]:
def gcd(a, b):
    """Compute gcd(a, b) using the Euclidean algorithm.
    Base case: gcd(a, 0) = a.
    Recursive step: gcd(a, b) = gcd(b, a mod b)."""
    while b != 0:
        a, b = b, a % b
    return a

# Sanity check against Python's built-in
test_pairs = [(48, 18), (252, 105), (151, 25), (391, 299), (99, 78), (64, 35)]
print('Comparing gcd() against math.gcd():')
for a, b in test_pairs:
    ours = gcd(a, b)
    theirs = math.gcd(a, b)
    match = '✓' if ours == theirs else '✗ MISMATCH'
    print(f'  gcd({a:>4}, {b:>4}) = {ours:<4}  {match}')

In [ ]:
def gcd_trace(a, b):
    """Run the Euclidean algorithm and print every step.
    Returns the gcd."""
    separator(f'gcd({a}, {b})')
    original_a, original_b = a, b
    step = 1
    while b != 0:
        q, r = divmod(a, b)
        print(f'  Step {step}: {a} = {q}·{b} + {r}  → gcd({a},{b}) = gcd({b},{r})')
        a, b = b, r
        step += 1
    print(f'  Step {step}: b = 0  → gcd({original_a}, {original_b}) = {a}')
    return a

gcd_trace(48, 18)
print()
gcd_trace(252, 105)

In [ ]:
def gcd_table(a, b):
    """Display the Euclidean algorithm in tabular format.
    Returns the gcd."""
    separator(f'gcd({a}, {b}) — table format')
    print(f'  {"Step":>5}  {"a":>8}  {"b":>8}  {"q":>5}  {"r":>8}')
    print(f'  {"─"*5}  {"─"*8}  {"─"*8}  {"─"*5}  {"─"*8}')
    step = 1
    while b != 0:
        q, r = divmod(a, b)
        print(f'  {step:>5}  {a:>8}  {b:>8}  {q:>5}  {r:>8}')
        a, b = b, r
        step += 1
    print(f'  {step:>5}  {a:>8}  {0:>8}  {"─":>5}  {"─":>8}  ← b=0, done')
    print(f'\n  Result: gcd = {a}')
    return a

gcd_table(391, 299)
print()
gcd_table(151, 25)

---
## Section 4 — Tutorial Examples Verified

Every worked example from Tutorial 06, confirmed step-by-step.

In [ ]:
# Tutorial examples in full trace format
examples = [
    (48,  18,   6),   # Example 1: 3 steps
    (252, 105, 21),   # Example 2: 3 steps
    (151,  25,  1),   # Example 3: 3 steps — coprime
    (391, 299, 23),   # Example 4: 3 steps
    (99,   78,  3),   # Practice 1
    (64,   35,  1),   # Practice 2
    (144,  89,  1),   # Practice 5 — Fibonacci chain, 10 steps
    (7,    26,  1),   # Practice 6 — affine cipher key validity
]

print('Tutorial 06 examples — verification summary:')
print(f'  {"(a, b)":>14}  {"expected":>8}  {"computed":>8}  {"match":>6}')
print(f'  {"─"*14}  {"─"*8}  {"─"*8}  {"─"*6}')
for a, b, expected in examples:
    computed = gcd(a, b)
    match = '✓' if computed == expected else '✗'
    print(f'  ({a:>4}, {b:>4})    {expected:>8}  {computed:>8}  {match:>6}')

In [ ]:
# Full trace for the Fibonacci pair (144, 89) — Tutorial 06 Practice 5
# This is the slowest-possible input for its size: 10 steps for numbers < 150
gcd_table(144, 89)

### Exercise 4.1

Trace `gcd(99, 78)` by hand (expect 3 steps, result = 3), then verify with `gcd_trace`.

Then find `gcd(7, 26)`. Why does the result matter for the affine cipher?

In [ ]:
# Your work here


---
## Section 5 — Large Numbers: Speed Demonstration

Trial division to find gcd(a, b) would test every integer up to min(a, b).
The Euclidean algorithm runs in **O(log min(a, b))** steps — exponentially faster.

In [ ]:
# Millisecond-scale computation on large numbers
large_pairs = [
    (1_234_567_890, 987_654_321),
    (10**18 + 9, 10**18 + 7),     # near-equal large numbers
    (2**128 - 159, 2**127 - 1),   # cryptographic-scale
]

separator('Euclidean algorithm on large numbers')
for a, b in large_pairs:
    start = time.perf_counter()
    result = math.gcd(a, b)
    elapsed = (time.perf_counter() - start) * 1_000_000   # microseconds
    digits_a = len(str(a))
    print(f'  gcd({digits_a}-digit, {len(str(b))}-digit) = {result}  [{elapsed:.1f} µs]')

In [ ]:
# Compare: how many steps does the algorithm take vs. how big are the numbers?
def gcd_steps(a, b):
    """Return the number of division steps needed to compute gcd(a, b)."""
    steps = 0
    while b != 0:
        a, b = b, a % b
        steps += 1
    return steps

separator('Step counts for large inputs')
print(f'  {"Input":>38}  {"Steps":>6}')
print(f'  {"─"*38}  {"─"*6}')
for a, b in large_pairs:
    steps = gcd_steps(a, b)
    label = f'gcd({len(str(a))}-digit, {len(str(b))}-digit)'
    print(f'  {label:>38}  {steps:>6}')

print()
print('  Even for 40-digit numbers, the algorithm needs fewer than ~140 steps.')

---
## Section 6 — Step Counts and the Fibonacci Worst Case

**Lamé's Theorem** (1844): the number of steps to compute gcd(*a*, *b*) is at most 5 times the number of decimal digits in the smaller number.

The **worst case** is consecutive Fibonacci numbers — each step subtracts one, giving the maximum number of steps possible for that size.

In [ ]:
def fibonacci(n):
    """Return the first n Fibonacci numbers: F(1)=1, F(2)=1, F(3)=2, ..."""
    fibs = [1, 1]
    for _ in range(n - 2):
        fibs.append(fibs[-1] + fibs[-2])
    return fibs

fibs = fibonacci(25)

separator('Fibonacci pairs — the worst case for step count')
print(f'  {"F(k+1)":>14}  {"F(k)":>14}  {"gcd":>5}  {"steps":>6}')
print(f'  {"─"*14}  {"─"*14}  {"─"*5}  {"─"*6}')
for k in range(4, 20):
    a, b = fibs[k], fibs[k-1]
    steps = gcd_steps(a, b)
    g = gcd(a, b)
    print(f'  {a:>14}  {b:>14}  {g:>5}  {steps:>6}')

In [ ]:
# Verify Lamé's bound: steps ≤ 5 * digits(min(a, b))
separator("Lamé's bound verification")
print(f'  {"(a, b)":>22}  {"steps":>6}  {"5·digits(b)":>12}  {"bound holds?":>13}')
print(f'  {"─"*22}  {"─"*6}  {"─"*12}  {"─"*13}')
for k in range(4, 20):
    a, b = fibs[k], fibs[k-1]
    steps = gcd_steps(a, b)
    digits_b = len(str(b))
    lame_bound = 5 * digits_b
    ok = '✓ Yes' if steps <= lame_bound else '✗ VIOLATION'
    print(f'  ({a:>9}, {b:>9})  {steps:>6}  {lame_bound:>12}  {ok:>13}')

### Exercise 6.1

1. For consecutive Fibonacci numbers F(k+1) and F(k), `gcd_steps` returns exactly k−1 steps. Verify this for k = 5, 8, 12.
2. Find a non-Fibonacci pair where the step count exceeds 5 (hint: try pairs near a Fibonacci number).

In [ ]:
# Your work here


---
## Section 7 — Valid Affine Cipher Keys Modulo 26

The affine cipher uses key (*a*, *b*) with encryption *C* ≡ *aP* + *b* (mod 26).
For decryption to be possible, *a* must have a multiplicative inverse mod 26 — meaning gcd(*a*, 26) = 1.

Tutorial 06 states there are exactly **12 valid multipliers**. Let's verify and list them all.

In [ ]:
separator('Valid affine multipliers modulo 26')

valid = []
invalid = []
print(f'  {"a":>4}  {"gcd(a,26)":>10}  {"valid?":>7}')
print(f'  {"─"*4}  {"─"*10}  {"─"*7}')
for a in range(1, 26):
    g = math.gcd(a, 26)
    is_valid = g == 1
    print(f'  {a:>4}  {g:>10}  {"Yes" if is_valid else "No":>7}')
    if is_valid:
        valid.append(a)
    else:
        invalid.append(a)

print(f'\n  Valid multipliers ({len(valid)}):  {valid}')
print(f'  Invalid multipliers ({len(invalid)}): {invalid}')
print(f'\n  Tutorial 06 states 12 valid multipliers: {"✓ confirmed" if len(valid) == 12 else "✗ mismatch"}')

In [ ]:
# Show the full key space: how many (a, b) affine cipher keys are there?
# a has 12 choices; b has 26 choices (0–25); total key space size:
total_keys = len(valid) * 26
print(f'Total affine cipher key space: {len(valid)} × 26 = {total_keys} keys')
print('(For comparison, Caesar cipher has only 26 keys.)')

### Exercise 7.1

1. Why are even values of *a* always invalid for the affine cipher mod 26?
2. Why is *a* = 13 specifically invalid?
3. Write a one-line list comprehension that generates all valid multipliers without calling `math.gcd`.

In [ ]:
# Your work here — answer questions 1 and 2 as comments, then write the list comprehension for 3


---
## Section 8 — RSA Preview: Coprimality at Scale

RSA encryption requires choosing a public exponent *e* that is coprime to φ(*n*) = (*p* − 1)(*q* − 1).
The Euclidean algorithm is how we verify this — and Module 07's extended version is how we compute *d* = *e*⁻¹ mod φ(*n*).

In [ ]:
# Toy RSA: small primes to see the structure clearly
p, q = 61, 53
n = p * q
phi_n = (p - 1) * (q - 1)
e = 17   # public exponent — must satisfy gcd(e, phi_n) = 1

separator('Toy RSA — coprimality check')
print(f'  Primes:    p = {p},  q = {q}')
print(f'  Modulus:   n = p × q = {n}')
print(f'  Totient:   φ(n) = (p−1)(q−1) = {p-1} × {q-1} = {phi_n}')
print(f'  Public e:  e = {e}')
print(f'  gcd(e, φ(n)) = gcd({e}, {phi_n}) = {math.gcd(e, phi_n)}')
print(f'  Valid e?   {"Yes — e is coprime to φ(n)" if math.gcd(e, phi_n) == 1 else "No — choose a different e"}')

In [ ]:
# The gcd trace shows exactly how we verify e is valid
gcd_trace(phi_n, e)

In [ ]:
# Python's pow(e, -1, phi_n) uses the Extended Euclidean Algorithm internally
# We'll implement that from scratch in Module 07
d = pow(e, -1, phi_n)   # private exponent

separator('RSA key derivation')
print(f'  Public key:  (e={e}, n={n})')
print(f'  Private key: (d={d}, n={n})')
print(f'  Verify:      e × d mod φ(n) = {e} × {d} mod {phi_n} = {(e * d) % phi_n}')

# Mini encrypt/decrypt round-trip
M = 42   # plaintext message (must be < n)
C = pow(M, e, n)   # C = M^e mod n
M2 = pow(C, d, n)  # M = C^d mod n
print(f'\n  Plaintext:   M = {M}')
print(f'  Encrypted:   C = M^e mod n = {M}^{e} mod {n} = {C}')
print(f'  Decrypted:   M = C^d mod n = {C}^{d} mod {n} = {M2}')
print(f'  Round-trip:  {"✓ M == M2" if M == M2 else "✗ FAILED"}')

In [ ]:
# Realistic RSA uses primes with hundreds of digits — gcd still runs in microseconds
import secrets

def is_prime_simple(n):
    """Simple primality test — fine for small numbers in examples."""
    if n < 2: return False
    if n < 4: return True
    if n % 2 == 0 or n % 3 == 0: return False
    i = 5
    while i * i <= n:
        if n % i == 0 or n % (i + 2) == 0:
            return False
        i += 6
    return True

# Find two 10-digit primes
def next_prime(start):
    n = start | 1   # make odd
    while not is_prime_simple(n):
        n += 2
    return n

p2 = next_prime(10**9 + 1)
q2 = next_prime(10**9 + 9999)
n2 = p2 * q2
phi_n2 = (p2 - 1) * (q2 - 1)
e2 = 65537   # standard RSA public exponent

separator('Larger RSA example — 10-digit primes')
print(f'  p = {p2}  (10 digits)')
print(f'  q = {q2}  (10 digits)')
print(f'  n = {n2}  (20 digits)')
print(f'  e = {e2},  gcd(e, φ(n)) = {math.gcd(e2, phi_n2)}')
print(f'  Steps to verify coprimality: {gcd_steps(phi_n2, e2)}')

### Exercise 8.1

For the toy RSA example (p=61, q=53, e=17):
1. Encrypt the message M=65 and decrypt it back.
2. What happens if you try to use e=3? Is gcd(3, φ(n)) = 1?

In [ ]:
# Your work here — use p=61, q=53 from above


---
## Section 9 — Exercises

### Exercise 9.1 — Build a GCD table for a range

Compute gcd(*a*, 30) for every *a* from 1 to 30. Print the results and count how many values are coprime to 30.

In [ ]:
# Your work here


### Exercise 9.2 — Find the slowest inputs below 100

Among all pairs (a, b) with 1 ≤ b < a ≤ 100, which pair requires the most steps?
Print the top 5 slowest pairs.

In [ ]:
# Your work here — use gcd_steps(a, b), sort by steps descending


### Exercise 9.3 — Visualise the step-count landscape

For each pair (a, b) with 1 ≤ b < a ≤ 30, build a 2-D grid showing the number of algorithm steps.
Where do you see the highest step counts?

In [ ]:
# Your work here — print a grid: row = a, column = b, cell = step count
# (Hint: use f'{gcd_steps(a,b):>3}' for fixed-width output)


---
## Section 10 — Summary and What Comes Next

### Tools built in this notebook

| Function | What it does |
|---|---|
| `gcd(a, b)` | Clean Euclidean algorithm — match the textbook pseudocode |
| `gcd_trace(a, b)` | Step-by-step trace: every division printed |
| `gcd_table(a, b)` | Tabular format matching Tutorial 06 worked examples |
| `gcd_steps(a, b)` | Count how many division steps the algorithm needs |
| `show_division(a, b)` | Display a single division step *a* = *q*·*b* + *r* |

### Key results confirmed

- gcd(48, 18) = 6 · gcd(252, 105) = 21 · gcd(151, 25) = 1 · gcd(391, 299) = 23
- Fibonacci pairs (F(k+1), F(k)) are the worst case — gcd_steps = k − 1
- Lamé's bound holds: steps ≤ 5 · digits(min(a, b)) for every pair tested
- Exactly **12** valid affine multipliers mod 26 (those coprime to 26)
- RSA key derivation reduces to a single gcd check — microseconds for 20-digit numbers

---
## What Comes Next

**Module 07 — The Extended Euclidean Algorithm** goes one step further:
instead of just computing gcd(*a*, *b*), it also finds integers *s* and *t* satisfying

$$as + bt = \gcd(a, b)$$

This is **Bézout's identity**, and when gcd(*a*, *n*) = 1 it gives *s* = *a*⁻¹ mod *n* directly.
That is how `pow(a, -1, n)` works under the hood — and how RSA computes the private key *d*.